# Episodic MoBA PPO (Colab launcher)

Thin launcher only: implementation/training logic stays in the repository. Run cells top-to-bottom; a failed baseline gate aborts the notebook.

In [4]:
from pathlib import Path
import os, subprocess
# Point these at the repository containing this retrofit, not the upstream base.
REPO_URL="https://github.com/getFaster/episodic-transformer-memory-ppo.git"
REPO_COMMIT="281acb7aa2efa94274906c938822a4ecc1b85138"
DRIVE_ROOT="/content/drive/MyDrive/episodic-moba-ppo"
arm="trxl_moba"; seed=1
REPO_DIR=Path("/content/episodic-moba-ppo")
BASE_CONFIG="configs/trxl_command40.yaml" if arm=="trxl" else "configs/trxl_moba_command40.yaml"
RUN_ID=f"{arm}-command40-seed{seed}"
RESOLVED_CONFIG=Path("/content")/f"{RUN_ID}.yaml"

In [16]:
def run(*args, cwd=REPO_DIR, env=None):
    e = os.environ.copy()
    e.update({k: str(v) for k, v in (env or {}).items()})

    print("$", " ".join(map(str, args)))

    try:
        return subprocess.run(
            [str(a) for a in args],
            cwd=cwd,
            env=e,
            check=True,
            text=True,
            capture_output=True,
        )
    except subprocess.CalledProcessError as exc:
        print("stdout:", exc.stdout)
        print("stderr:", exc.stderr)
        raise

In [6]:
if REPO_URL.startswith("REPLACE_") or REPO_COMMIT.startswith("REPLACE_"):
    raise ValueError("Set REPO_URL and REPO_COMMIT to the published implementation revision")
if not REPO_DIR.exists(): run("git","clone",REPO_URL,REPO_DIR,cwd=Path("/content"))
run("git","fetch","--tags","--force")
run("git","checkout","--detach",REPO_COMMIT)
actual=subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO_DIR,text=True).strip()
assert actual == REPO_COMMIT, (actual, REPO_COMMIT)


$ git fetch --tags --force
$ git checkout --detach 281acb7aa2efa94274906c938822a4ecc1b85138


In [7]:
from google.colab import drive
drive.mount("/content/drive");
Path(DRIVE_ROOT).mkdir(parents=True,exist_ok=True)
run("python","-m","pip","install","-q","uv")
run("uv","python","install","3.11")
run("uv","sync","--frozen","--python","3.11")


Mounted at /content/drive
$ python -m pip install -q uv
$ uv python install 3.11
$ uv sync --frozen --python 3.11


CompletedProcess(args=['uv', 'sync', '--frozen', '--python', '3.11'], returncode=0, stdout='', stderr='Using CPython 3.11.16\nCreating virtual environment at: .venv\nPrepared 75 packages in 1m 20s\nInstalled 75 packages in 738ms\n + accelerate==1.15.0\n + annotated-doc==0.0.5\n + annotated-types==0.8.0\n + anyio==4.15.1\n + certifi==2026.7.22\n + charset-normalizer==3.5.1\n + click==8.5.0\n + cloudpickle==3.1.2\n + einops==0.8.1\n + episodic-moba-ppo==0.1.0 (from file:///content/episodic-moba-ppo)\n + farama-notifications==0.0.6\n + filelock==3.32.6\n + fsspec==2026.7.0\n + gitdb==4.0.12\n + gitpython==3.1.62\n + gymnasium==0.29.0\n + h11==0.16.0\n + hf-xet==1.6.0\n + httpcore==1.0.9\n + httpx==0.28.1\n + huggingface-hub==1.31.0\n + idna==3.19\n + jinja2==3.1.6\n + markdown-it-py==4.2.0\n + markupsafe==3.0.3\n + mdurl==0.1.2\n + memory-gym==1.0.2\n + mpmath==1.3.0\n + networkx==3.6.1\n + numpy==1.26.4\n + nvidia-cublas-cu12==12.8.4.1\n + nvidia-cuda-cupti-cu12==12.8.90\n + nvidia-cuda-

In [8]:
!uv pip install wandb==0.30.0

Using Python 3.13.15 environment at: /usr
Resolved 24 packages in 589ms
Prepared 7 packages in 805ms
Uninstalled 4 packages in 132ms
Installed 7 packages in 57ms
 - opentelemetry-api==1.42.1
 + opentelemetry-api==1.44.0
 + opentelemetry-exporter-otlp-proto-common==1.44.0
 + opentelemetry-exporter-otlp-proto-http==1.44.0
 + opentelemetry-proto==1.44.0
 - opentelemetry-sdk==1.42.1
 + opentelemetry-sdk==1.44.0
 - opentelemetry-semantic-conventions==0.63b1
 + opentelemetry-semantic-conventions==0.65b0
 - wandb==0.28.1
 + wandb==0.30.0


In [9]:
import os
import wandb
from google.colab import userdata

secret_key = userdata.get('WANDB_API_KEY')
if secret_key:
    os.environ['WANDB_API_KEY'] = secret_key
    print("WANDB_API_KEY retrieved and set as environment variable")
else:
    assert False

wandb.login()

WANDB_API_KEY retrieved and set as environment variable


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: kevinwu777k (kevinwu777k-national-taiwan-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Mandatory baseline gate

The command exits non-zero if either threshold fails; `run` propagates that failure and prevents later cells from running.

## Resolve, train, resume, evaluate, analyze

Create a run-specific YAML without mutating checked-in configs. Resume only from a durable update directory containing `commit_success.json`.


In [18]:
print(wandb.__version__)

0.30.0


In [19]:
r = wandb.init(
    project="memrl-wandb-test",
    entity="kevinwu777k-national-taiwan-university",
)
r.log({"test": 1})
r.finish()

test,▁
test,1


In [20]:
run(
    "uv", "run", "python", "-c",
    "import sys, wandb; print(sys.executable); print(sys.version); print(wandb.__version__)"
)

$ uv run python -c import sys, wandb; print(sys.executable); print(sys.version); print(wandb.__version__)


CompletedProcess(args=['uv', 'run', 'python', '-c', 'import sys, wandb; print(sys.executable); print(sys.version); print(wandb.__version__)'], returncode=0, stdout='/content/episodic-moba-ppo/.venv/bin/python\n3.11.16 (main, Sep  1 2026, 14:18:37) [Clang 22.1.3 ]\n0.21.3\n', stderr='')

In [21]:
run("uv", "add", "wandb==0.30.0")

$ uv add wandb==0.30.0


CompletedProcess(args=['uv', 'add', 'wandb==0.30.0'], returncode=0, stdout='', stderr='Resolved 95 packages in 966ms\nPrepared 3 packages in 1.45s\nUninstalled 2 packages in 37ms\nInstalled 10 packages in 28ms\n ~ episodic-moba-ppo==0.1.0 (from file:///content/episodic-moba-ppo)\n + googleapis-common-protos==1.75.3\n + opentelemetry-api==1.44.0\n + opentelemetry-exporter-otlp-proto-common==1.44.0\n + opentelemetry-exporter-otlp-proto-http==1.44.0\n + opentelemetry-proto==1.44.0\n + opentelemetry-sdk==1.44.0\n + opentelemetry-semantic-conventions==0.65b0\n - wandb==0.21.3\n + wandb==0.30.0\n + xxhash==4.0.1\n')

In [ ]:
import yaml
with (REPO_DIR/BASE_CONFIG).open() as f: cfg=yaml.safe_load(f)
cfg["wandb"]["entity"] = "kevinwu777k-national-taiwan-university"
cfg["seeds"]["model"]=seed
cfg["wandb"]["run_name"]=RUN_ID
cfg["drive"]["root"]=DRIVE_ROOT
cfg["checkpointing"]["local_dir"]=f"/content/checkpoints/{RUN_ID}"
cfg["checkpointing"]["drive_dir"]=f"checkpoints/{RUN_ID}"
with RESOLVED_CONFIG.open("w") as f:
  yaml.safe_dump(cfg,f,sort_keys=False)

run(
    "uv", "run", "train",
    "--config", RESOLVED_CONFIG,
    "--repo-root", ".",
    env={"PYTHONHASHSEED": seed},
)

$ uv run train --config /content/trxl_moba-command40-seed1.yaml --repo-root .


In [ ]:
# Use update-00062 only after the shared extension gate authorizes both arms.
CHECKPOINT_DIR=Path(DRIVE_ROOT)/"checkpoints"/RUN_ID/"update-00031"
EVALUATION_OUTPUT=REPO_DIR/"results"/f"{RUN_ID}.json"
run("uv","run","evaluate","--checkpoint",CHECKPOINT_DIR,"--arm",arm,"--model-seed",seed,"--output",EVALUATION_OUTPUT)
if arm == "trxl_moba":
    run("uv","run","analyze-retrieval","--config","configs/analyze_retrieval.yaml")
